# Section 2 — Supervised multi-label classification

In [14]:
import numpy as np
import pandas as pd
import seaborn as sns

from src import data
from src import supervised

import matplotlib.pyplot as plt

DATA_PATH = "data/ssh_attacks.parquet"

# Reading the parquet file... same as 01_data.ipynb
try: 
    df = data.load_dataset(DATA_PATH)

    # Empty rows and empty columns
    print(f"Empty rows: {df.isna().all(axis=1).sum()}")
    print(f"Empty columns: {df.isna().all(axis=0).sum()}")

except Exception :
    print(f"An ERROR occurred while reading the dataset")

finally:
    pass


Empty rows: 0
Empty columns: 0


In [18]:
from sklearn.preprocessing import MultiLabelBinarizer

# Same as in section 1 : the set_fingerprint is mapped in a matrix of 0/1.
mlb = MultiLabelBinarizer(classes=data.LABELS)
y = mlb.fit_transform(df["Set_Fingerprint"])

# The columns of X are := [text, timestamp, ID]
# In the training part REMEMBER to choose the interesting columns.
X = df.drop(columns=["Set_Fingerprint"])

print(f"X: {X.shape[0]} sessions x {X.shape[1]} columns {list(X.columns)}")
print(f"y: {y.shape[0]} sessions x {y.shape[1]} labels {list(mlb.classes_)}")

X: 233035 sessions x 3 columns ['session_id', 'full_session', 'first_timestamp']
y: 233035 sessions x 7 labels ['Persistence', 'Discovery', 'Defense Evasion', 'Execution', 'Impact', 'Other', 'Harmless']


## Section 2.1

Compare random validation with two alternatives: chronological validation and separation of text-template groups. Define splits without using labels to construct similarity groups. Inspect overlap and per-label support; explain which evaluation matches a possible deployment of the classifier. 
Discuss the validation strategies with the teacher.

_RANDOM SPLITTING_

In [19]:
from sklearn.model_selection import train_test_split

# 60% train, 20% validation, 20% test.
X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, train_size=0.60, random_state=data.SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, train_size=0.50, random_state=data.SEED)


sizes = pd.Series({"train": len(X_train), "validation": len(X_val), "test": len(X_test)})
display(pd.DataFrame({"sessions": sizes, "% of dataset": (100 * sizes / len(df)).round(1)}))


,sessions,% of dataset
train,139821,60.0
validation,46607,20.0
test,46607,20.0


In [20]:
# for each label, how many sessions have that label in each partition.
print("for each label, how many sessions have that label in each partition (random split):")
support = pd.DataFrame({"train": y_train.sum(axis=0),
                        "validation": y_val.sum(axis=0),
                        "test": y_test.sum(axis=0)}, index=data.LABELS)
display(support)

# Same numbers as % of the sessions of each partition:
# with the random split every partition has about the same label mix.
share = (100 * support / sizes).round(2)
print("% of the sessions of each partition with the label (random split):")
display(share)

for each label, how many sessions have that label in each partition (random split):


,train,validation,test
Persistence,126711,42323,42261
Discovery,139267,46454,46424
Defense Evasion,11467,3723,3809
Execution,56036,18317,18574
Impact,18,5,4
Other,182,70,75
Harmless,1330,439,437


% of the sessions of each partition with the label (random split):


,train,validation,test
Persistence,90.62,90.81,90.68
Discovery,99.60,99.67,99.61
Defense Evasion,8.20,7.99,8.17
Execution,40.08,39.30,39.85
Impact,0.01,0.01,0.01
Other,0.13,0.15,0.16
Harmless,0.95,0.94,0.94


In [ ]:
# Overlap (random split) -> share of validation/test sessions whose template also appears in train
groups = data.template_groups(df["full_session"])      
    # group of each session that share the same index as df
train_groups = set(groups[X_train.index])              
    # groups that have at least one session in train
for name, part in [("validation", X_val), ("test", X_test)]:
    seen = groups[part.index].isin(train_groups)
    print(f"{name}: {seen.mean():.1%} of sessions have a template seen in train")

validation: 99.7% of sessions have a template seen in train
test: 99.7% of sessions have a template seen in train


For the random split we used the 60% of the dataSet for the training, the 20% of the dataSet for the validation and the remaining 20% for the test.
All sessions are drawn at random so every partition has about the same share of each label

- As we saw "Impact" has only 18 / 5 / 4 positives in train / validation / test, so its metrics will be unreliable. "Other" (182 / 70 / 75) is better but still small. The split is not stratified, so the number of positives of the rare labels depends on the random draw.

- Most sessions are the same script run many times as we saw in Section 1.5, so with a random split almost every validation and test session has a session with the same template in train: 99.7% of the validation and test sessions. The model is evaluated almost only on variants of scripts it has already seen, so this evaluation is optimistic.

- The partitions mix all the periods of the dataset: train also contains sessions recorded after some test sessions, which is not possible in a real deployment, this is the reason why the chronological splitting will be a more suitable method to represent the reality.

The random split can be considered as an upper reference: comparing it with the chronological and the group splits shows how much the performance depends on sessions that are almost identical to the training ones.

_CHRONOLOGICAL VALIDATION_

In [ ]:
# first_timestamp is stored as text so we convert it to a date 
# .assign returns a copy of the whole dataset X + the new column 'first_timestamp'.
X_time = X.assign(first_timestamp = pd.to_datetime(X["first_timestamp"], utc=True))

# y is a numpy array so we can order via iloc / indexing.
order = X_time["first_timestamp"].argsort().to_numpy()
X_sorted, y_sorted = X_time.iloc[order], y[order]

# shuffle=False: train_test_split cuts the rows in their chronological order.
X_train_t, X_rest_t, y_train_t, y_rest_t = train_test_split(
    X_sorted, y_sorted, train_size=0.60, shuffle=False)
X_val_t, X_test_t, y_val_t, y_test_t = train_test_split(
    X_rest_t, y_rest_t, train_size=0.50, shuffle=False)

# Checking for missing or repeated timestamps.
print(f"Missing timestamps: {X_time['first_timestamp'].isna().sum()}, "
      f"repeated timestamps: {X_time['first_timestamp'].duplicated().sum()}")

# Period covered by each partition
periods = pd.DataFrame({
    name: {"sessions": len(part),
           "% of dataset": round(100 * len(part) / len(df), 1),
           "from": part["first_timestamp"].min(),
           "to": part["first_timestamp"].max()}
    for name, part in [("train", X_train_t), ("validation", X_val_t), ("test", X_test_t)]
}).T
display(periods)

# We could check if the chronological order of the sets if mantained.
# assert X_train_t["first_timestamp"].max() < X_val_t["first_timestamp"].min()
# assert X_val_t["first_timestamp"].max() < X_test_t["first_timestamp"].min()

Missing timestamps: 0, repeated timestamps: 0


,sessions,% of dataset,from,to
train,139821,60.0,2019-06-04 09:45:11.151186+00:00,2019-12-08 19:16:50.407505+00:00
validation,46607,20.0,2019-12-08 19:16:54.665860+00:00,2019-12-30 17:21:34.006723+00:00
test,46607,20.0,2019-12-30 17:27:47.782221+00:00,2020-02-29 23:59:22.199490+00:00


In [23]:
# for each label, how many sessions have that label in each partition (chronological split)
support_t = pd.DataFrame({"train": y_train_t.sum(axis=0),
                          "validation": y_val_t.sum(axis=0),
                          "test": y_test_t.sum(axis=0)}, index=data.LABELS)
print("for each label, how many sessions have that label in each partition (chronological split):")
display(support_t)

# Same numbers as % of the sessions of each partition:
# with the chronological split the mix changes over time.
share_t = (100 * support_t / support_t.columns.map(
    {"train": len(X_train_t), "validation": len(X_val_t), "test": len(X_test_t)})).round(2)
print("% of the sessions of each partition with the label (chronological split):")
display(share_t)

for each label, how many sessions have that label in each partition (chronological split):


,train,validation,test
Persistence,121009,44479,45807
Discovery,139039,46590,46516
Defense Evasion,18365,496,138
Execution,87496,4907,524
Impact,15,4,8
Other,319,1,7
Harmless,1029,750,427


% of the sessions of each partition with the label (chronological split):


,train,validation,test
Persistence,86.55,95.43,98.28
Discovery,99.44,99.96,99.80
Defense Evasion,13.13,1.06,0.30
Execution,62.58,10.53,1.12
Impact,0.01,0.01,0.02
Other,0.23,0.00,0.02
Harmless,0.74,1.61,0.92


In [ ]:
# Overlap (chronological split) -> share of validation/test sessions whose template also appears in train
train_groups_t = set(groups[X_train_t.index])
for name, part in [("validation", X_val_t), ("test", X_test_t)]:
    seen = groups[part.index].isin(train_groups_t)
    print(f"{name}: {seen.mean():.1%} of sessions have a template seen in train")

validation: 95.7% of sessions have a template seen in train
test: 98.5% of sessions have a template seen in train


For the chronological split we also used the 60% of the dataSet for the training, the 20% of the dataSet for the validation and the remaining 20% for the test.

- The population is not well distributed and changes over time. "Defense Evasion" has about 18,000 sessions in train but only a few hundred in validation and test, such as "Execution"'s distribution which is unbalanced too. "Other" almost disappears after October 2019. The model will be evaluated on a different label mix from the one it learned on.

- The honeypot upgrade (November 8, 2019) is entirely inside train time, this means that the validation and test contain only sessions recorded after the upgrade.

- Even with the chronological split, 95.7% of the validation sessions and 98.5% of the test sessions have a template already seen in train. This happens beacuse as we saw in Section 1.5 large scripts keep running for months (first_seen - last_seen), so the same templates appear both in the past and in the future. Splitting by time changes the label mix, but it does not remove the long term duplicates. Only the split by template groups can evaluate the model on scripts it has never seen.

- Even if this split seems to be worse it's the one that best matches a real deployment: you train on what you've collected so far and classify the sessions that arrive afterwards. That's why we expect lower scores than with the random split.

TOASK: How to divide the groups? The first template group represents about the 50.9% of the whole dataset, this means having a training set made manly of ONE group

## Section 2.2